# Chapter 9: Retrieval — Putting It to Work
## Topic 1: Retrieval Triggers — Deciding *When* to Retrieve

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Chapters 7 and 8 built an excellent retrieval-and-generation pipeline — but they both implicitly assumed retrieval should *always* run for every incoming query. That assumption is wrong for a real production system, and this topic exists to fix it.
- Not every email needs retrieval. A rule-based classifier can already handle unambiguous cases (a login issue, a clear loan complaint) with no need to search a Fixed Deposit knowledge base at all. Running full hybrid retrieval, reranking, and grounded generation on every single email regardless of its content wastes latency and cost on cases where it adds no value, and in the worst case, actively *invites* a wrong answer by retrieving and citing irrelevant content just because the pipeline was forced to produce something.
- The core reframe this topic introduces: retrieval is not a mandatory first step — it's a **decision**. Something upstream (a rule, a classifier, a confidence gate, or the model itself deciding) must determine whether this specific query actually benefits from retrieval before the expensive retrieval-and-generation machinery runs at all.
- This connects directly to the project's existing rules → ML → confidence-gate → GenAI → tools pipeline design: retrieval triggering is precisely the decision of *where in that chain a query should enter*, and whether it needs to reach the GenAI/retrieval stage at all.


### 2. Internal Working — Step by Step

**Three practical trigger strategies, in increasing sophistication:**

1. **Rule-based triggering (cheapest, first line of defense):** if the email was already classified as "Non-FD" by the existing keyword-rule engine (Chapter 1's `classify_by_keyword_rules()`), retrieval never runs at all — there's nothing FD-related to look up. This is a pure routing decision made *before* any model call, using signal that's already free and available.
2. **Confidence-gated triggering:** if an upstream classifier (rules or ML) produces a low-confidence or ambiguous verdict, this is itself a signal that the query may need deeper handling — trigger retrieval specifically to gather grounding context that could help a subsequent GenAI call resolve the ambiguity, rather than triggering retrieval on every single query regardless of whether the simpler layers already handled it confidently.
3. **Model-decided triggering (the most flexible, and most expensive):** give the model itself a tool it can choose to call — `search_knowledge_base` — and let it decide, based on the specific content of the query, whether looking something up is actually necessary. A model asked "what's your name" doesn't need to call a retrieval tool; a model asked "what is the exact penalty percentage for my FD" should. This is the foundation for Agentic RAG (Chapter 13), where retrieval becomes just one of several tools an agent can choose to invoke rather than an unconditional pipeline stage.

**The decision sits upstream of retrieval, not inside it:** none of these three strategies change how retrieval itself works once triggered — Chapter 7's hybrid BM25+dense+RRF, reranking, and MMR remain exactly as built. What changes is whether that machinery gets invoked *at all* for a given query.


### 3. How This Is Implemented in This Project

- The existing pipeline already has a natural triage layer: `classify_by_keyword_rules()` produces one of FD / Non-FD / Multiple Category / ambiguous. Retrieval triggering slots in immediately after this step:
  - Non-FD verdicts skip retrieval entirely — there's no FD knowledge to look up, and forcing a retrieval call here would waste cost and risk retrieving spuriously "related" chunks that have nothing to do with the actual (non-FD) topic.
  - Clear FD verdicts with high confidence *could* still skip full retrieval if the query matches a very narrow, already-known FAQ pattern with a cached answer — but for anything beyond the simplest fixed patterns, FD verdicts are exactly the case retrieval exists to serve.
  - Ambiguous or Multiple Category verdicts are the strongest trigger signal — this is precisely the case where the rule engine's structural weakness (Chapter 7 Topic 5's finding that it can't distinguish "mentioned once" from "the whole point") means additional grounding context could help a downstream GenAI call make a better-informed decision.
- For the model-decided approach, `search_knowledge_base` is exposed as a standard tool in the same tool-calling pattern already used elsewhere in the project (Chapter 3's `run_agent()`/`TOOLS` pattern) — the model sees the tool's description and decides whether to call it, exactly like any other tool.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Under-triggering (missing a case that needed retrieval):** if the rule-based trigger is too conservative — for example, treating a borderline "Multiple Category" case as not worth retrieval — a genuinely FD-related query might get a shallow, ungrounded answer with no chance for citation or verification. This is a silent failure mode: nothing crashes, the answer is just quietly worse than it could have been.
- **Over-triggering (retrieval running when it isn't needed):** the opposite failure wastes latency and cost on every query, and in rare cases can actively hurt answer quality — the retrieval and reranking pipeline being forced to return *something* for a query that had no genuinely relevant content in the knowledge base at all, that "something" being a moderately-scored but ultimately irrelevant chunk that then gets cited and treated as if it were meaningfully grounding the answer.
- **Model-decided triggering's own reliability risk:** letting the model decide whether to call `search_knowledge_base` inherits all of instruction-following's probabilistic nature — a model might skip calling the tool for a query that genuinely needed it, or call it unnecessarily for a query it could have answered correctly without any lookup. This should be measured, not assumed reliable, exactly the same evidence-based discipline used everywhere else in this project.
- **Monitoring:** track the trigger rate (what fraction of queries actually invoke retrieval) broken down by the upstream classification verdict, and specifically watch for any drift where a growing fraction of "ambiguous" or "Multiple Category" queries are *not* triggering retrieval — this is a leading indicator of a triggering-logic regression, not just a retrieval-quality regression.
- **Cost and latency:** retrieval triggering is the single cheapest lever available for controlling the project's overall per-email cost — a query correctly routed away from retrieval and generation entirely costs essentially nothing beyond the initial classification step, while a query routed through full retrieval and grounded generation costs the entire pipeline's combined cost. At production volume, getting this routing decision right has a larger cost impact than almost any optimization within the retrieval pipeline itself.
- **Security:** a model deciding for itself whether to call `search_knowledge_base` introduces a subtle injection consideration — adversarial content in a query could attempt to manipulate the model into either avoiding a lookup it should make (hiding relevant grounding) or invoking it excessively as a way to probe what content exists in the knowledge base. This deserves the same "treat email content as data, not commands" discipline established elsewhere in the project.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Rule-based vs confidence-gated vs model-decided triggering:** rule-based triggering is cheapest and most predictable but can only encode patterns a human explicitly anticipated. Confidence-gated triggering adapts to the upstream classifier's own uncertainty, which is more flexible but depends on that classifier's confidence signal actually being well-calibrated. Model-decided triggering is the most flexible and handles genuinely novel query patterns best, but is also the least predictable and carries real cost at scale if the model over-triggers.
- **Where in the pipeline should the trigger decision sit?** The earliest, cheapest possible point is best — deciding *before* any model call, using free signal already computed (like the rule engine's verdict), avoids paying for a model call just to decide whether to make a further, more expensive model call. Only genuinely ambiguous cases that the cheap signal can't resolve should escalate to a model-decided trigger.
- **The real trade-off in a hybrid approach (using both rule-based pre-filtering AND model-decided triggering for what gets through):** rule-based pre-filtering handles the bulk of clearly-decidable cases at near-zero cost, and model-decided triggering is reserved only for the harder cases that make it past that first filter — this combines the cost-efficiency of rules with the flexibility of model judgment, applied only where it's actually needed.


### 6. Alternatives and When to Use Each

- **Always retrieve (no triggering logic at all):** the simplest possible design, but wastes cost and latency on every query regardless of whether retrieval is needed — reasonable only very early in a project, before enough real traffic exists to design meaningful triggering rules.
- **Rule-based triggering:** the right choice for the large fraction of queries that already fall into a clearly-decidable category via existing classification signal — cheap, fast, and predictable.
- **Confidence-gated triggering:** appropriate when the upstream classifier produces a genuinely meaningful, well-calibrated confidence score, and ambiguous cases are common enough to justify a dedicated escalation path.
- **Model-decided triggering (tool-based):** the right choice for handling genuinely novel or hard-to-anticipate query patterns, and the natural foundation for a more general agentic architecture — but should be reserved for cases that survive the cheaper upstream filters, not applied to every query uniformly.


### 7. Common Mistakes and Production Failures

- Running retrieval unconditionally on every single query, ignoring cheap upstream signal (like an existing rule-based classification) that already indicates retrieval isn't needed — a direct, ongoing cost and latency tax with no corresponding benefit for a large fraction of traffic.
- Under-triggering on ambiguous or Multiple Category cases specifically — this is exactly the case where additional grounding context is most likely to help, and skipping retrieval here silently produces weaker answers precisely where the pipeline needed the most help.
- Trusting model-decided triggering without measuring whether the model actually calls the retrieval tool reliably when it should — instruction-following for tool invocation is just as probabilistic as instruction-following for any other behavior, and needs the same evidence-based verification discipline.
- Not monitoring the trigger rate broken down by upstream classification verdict, missing an early signal of a triggering-logic regression before it shows up as a broader answer-quality problem.


### 8. Lead-Level Interview Questions

**Basic**

- Q: Why shouldn't retrieval run unconditionally on every incoming query?
  A: A meaningful fraction of queries can already be confidently handled by cheaper upstream signal, like an existing rule-based classifier — running full retrieval and grounded generation on these wastes latency and cost with no corresponding quality benefit, and can even introduce risk if the retrieval pipeline is forced to return irrelevant content just because it was invoked unconditionally.

- Q: What are the three practical strategies for deciding when to trigger retrieval?
  A: Rule-based triggering (using existing cheap classification signal to skip retrieval entirely for clearly-decidable cases), confidence-gated triggering (escalating to retrieval specifically when an upstream classifier is uncertain), and model-decided triggering (exposing retrieval as a tool the model itself chooses to invoke based on the specific query).

**Intermediate**

- Q: Where should the retrieval-trigger decision sit in the pipeline, and why does that placement matter for cost?
  A: It should sit as early and as cheaply as possible — ideally using signal that's already computed for free, like an existing rule-based classifier's verdict, before any model call is made. Placing the decision late, or making it via an expensive model call for every query, means paying real cost just to decide whether to pay further cost, defeating much of the point of having a cheap triggering layer at all.

- Q: How would you validate that model-decided triggering (via a tool the model can call) is actually reliable?
  A: Treat it exactly like any other measured behavior — build a labeled evaluation set of queries that genuinely do and don't need retrieval, run them through the model-decided triggering mechanism, and measure how often the model's tool-call decision matches the correct decision. This is the same evidence-based discipline used for tuning any other threshold or behavior in the pipeline, not something to assume works correctly by default.

**Advanced**

- Q: Design a layered triggering strategy that combines rule-based, confidence-gated, and model-decided approaches.
  A: Start with rule-based triggering as the first, cheapest filter — clearly Non-FD cases skip retrieval entirely, using signal already available from existing classification. For cases that pass this filter, use the classifier's confidence score as a second gate — high-confidence FD cases may proceed directly to retrieval, since the case is unambiguous. Genuinely ambiguous or Multiple Category cases, which the cheap layers can't resolve confidently, escalate to a model-decided mechanism where retrieval is exposed as a tool the model can choose to invoke based on the specific query content. This concentrates the most expensive, least predictable decision-making layer only on the subset of traffic that actually needs it.

- Q: A teammate argues that model-decided triggering alone is sufficient and the cheaper rule-based/confidence-gated layers are redundant complexity. How do you respond?
  A: Model-decided triggering, while flexible, still requires a full model call just to make the triggering decision — for the large fraction of traffic that a cheap rule-based classifier can already confidently resolve, this is pure added cost and latency with no corresponding benefit. The cheaper layers aren't redundant; they're what makes the expensive, flexible layer economically viable at scale, by ensuring it only runs on the harder cases that actually need its flexibility.

**Scenario-based**

- Q: Production monitoring shows retrieval is being triggered for nearly 100% of incoming queries, even ones the rule-based classifier confidently marks as Non-FD. Diagnose.
  A: This strongly suggests the triggering logic isn't actually consulting the rule-based classifier's verdict before deciding to retrieve — either a wiring bug where the classification result isn't being checked at all before the retrieval call, or the triggering logic was implemented as "always retrieve" and never actually updated to use the cheaper upstream signal. The fix is ensuring the routing decision genuinely happens before retrieval, using the classifier's verdict as the first, cheapest gate.

**Follow-up questions to expect:**

- "How would you measure the cost savings from proper retrieval triggering at your project's production volume?"
- "What's the risk of under-triggering versus over-triggering, and which is worse for this specific domain?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **Retrieval triggering is a specific instance of a general "should I do the expensive thing" pattern:** many production ML and AI systems face this same decision — a cheap, fast path handles the easy majority of cases, and a slower, more expensive path is reserved for the harder minority. Recognizing this as a general pattern (sometimes called a cascade or gating architecture) makes the specific retrieval-triggering decision easier to reason about.
- **This topic is the conceptual seed for Agentic RAG (Chapter 13):** model-decided triggering, where the model itself chooses whether to call a retrieval tool, is exactly the mechanism that Agentic RAG formalizes and extends — understanding retrieval triggering deeply here makes that later topic much faster to grasp.
- **Triggering decisions compound with everything downstream:** a wrong triggering decision doesn't just affect this one step — it silently determines whether citation, faithfulness checking, and hallucination detection (Chapter 8) even get a chance to operate at all for a given query, since none of that infrastructure matters if retrieval was never triggered in the first place.

### 10. Quick Revision Summary (for last-minute interview prep)

> Retrieval triggering reframes retrieval from a mandatory first step into a deliberate decision — should this specific query even invoke the retrieval-and-generation machinery at all? Rule-based triggering uses cheap, already-available classification signal to skip retrieval entirely for clearly-decidable cases, at near-zero added cost. Confidence-gated triggering escalates specifically when an upstream classifier is uncertain, targeting retrieval's benefit at genuinely ambiguous cases. Model-decided triggering exposes retrieval as a tool the model itself can choose to invoke, offering the most flexibility for novel query patterns at the highest cost and least predictability. The right production architecture layers these together — cheap rule-based filtering first, model-decided flexibility reserved only for what survives that filter — because getting this routing decision right has a larger cost impact at scale than almost any optimization within the retrieval pipeline itself, and a wrong triggering decision silently determines whether every downstream verification mechanism (citation, faithfulness, hallucination detection) ever gets a chance to run at all.


### Module 1: The Existing Rule-Based Classifier (Reused as the First Trigger Gate)

Rebuild the Chapter 1 keyword classifier — this becomes the free, first-line signal that decides whether retrieval is even worth considering.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Existing rule-based classifier, reused as trigger gate
# ------------------------------------------------------------------

FD_KEYWORD_GROUPS = [
    ["fd", "fixed deposit", "maturity", "machurity", "byaj"],
    ["premature", "withdrawal", "nikalna"],
    ["nominee", "interest rate", "tenure"],
]
NEGATION_PHRASES = ["loan", "emi", "insurance", "card", "login", "otp", "kyc"]


def classify_by_keyword_rules(text: str) -> str:
    """The existing Chapter 1 rule-based classifier -- pure binary
    keyword presence, no model call, essentially free to run."""
    text = text.lower()
    has_fd_term = any(kw in text for group in FD_KEYWORD_GROUPS for kw in group)
    has_negation = any(neg in text for neg in NEGATION_PHRASES)
    if has_fd_term and not has_negation:
        return "FD"
    elif has_negation and not has_fd_term:
        return "Non-FD"
    elif has_fd_term and has_negation:
        return "Multiple Category"
    else:
        return "ambiguous"


TEST_EMAILS = [
    ("Clear FD question", "What is the penalty for premature FD withdrawal?"),
    ("Clear Non-FD complaint", "My personal loan EMI bounced twice this month, please fix this."),
    ("Ambiguous mixed case", "I applied for a loan against my FD, need urgent help please."),
    ("Vague/no signal", "Hi, I need some help with my account please respond soon."),
    ("Clear FD, senior citizen rate", "What is the FD interest rate for senior citizens?"),
]

print("=" * 70)
print("MODULE 1: RULE-BASED CLASSIFICATION (the free first gate)")
print("=" * 70)
for label, email in TEST_EMAILS:
    verdict = classify_by_keyword_rules(email)
    print(f"  [{label:<28}] -> {verdict:<18} | \"{email[:50]}...\"")

print("\nThis classification is ESSENTIALLY FREE -- no model call, pure")
print("string matching. Module 2 uses this verdict to decide whether")
print("retrieval is even worth considering, before any expensive call.")
print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: RULE-BASED CLASSIFICATION (the free first gate)
  [Clear FD question           ] -> FD                 | "What is the penalty for premature FD withdrawal?..."
  [Clear Non-FD complaint      ] -> Non-FD             | "My personal loan EMI bounced twice this month, ple..."
  [Ambiguous mixed case        ] -> Multiple Category  | "I applied for a loan against my FD, need urgent he..."
  [Vague/no signal             ] -> ambiguous          | "Hi, I need some help with my account please respon..."
  [Clear FD, senior citizen rate] -> FD                 | "What is the FD interest rate for senior citizens?..."

This classification is ESSENTIALLY FREE -- no model call, pure
string matching. Module 2 uses this verdict to decide whether
retrieval is even worth considering, before any expensive call.

Module 1 complete. Run Module 2 next.


### Module 2: The Layered Triggering Decision

Implement the actual three-tier decision: rule-based skip, confidence-gated escalation, and model-decided flag for the hardest residual cases.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Layered retrieval-triggering decision
# ------------------------------------------------------------------

def should_trigger_retrieval(email_text: str) -> dict:
    """Implements the layered triggering strategy from the theory:
    1. Rule-based gate: Non-FD verdicts skip retrieval entirely (free).
    2. Confidence-gated escalation: ambiguous/Multiple Category verdicts
       are treated as needing retrieval, since this is exactly where
       grounding context helps most.
    3. Clear FD verdicts proceed to retrieval directly (the case
       retrieval exists to serve).
    Returns a full decision trace for transparency/debugging."""

    verdict = classify_by_keyword_rules(email_text)

    if verdict == "Non-FD":
        return {
            "trigger_retrieval": False,
            "reason": "Rule-based gate: classified Non-FD, no FD content to look up.",
            "cost_tier": "FREE (rule-based only, no model call)",
            "verdict": verdict,
        }
    elif verdict == "FD":
        return {
            "trigger_retrieval": True,
            "reason": "Rule-based gate: clear FD verdict, retrieval directly serves this case.",
            "cost_tier": "FULL RETRIEVAL + GENERATION",
            "verdict": verdict,
        }
    else:  # "ambiguous" or "Multiple Category"
        return {
            "trigger_retrieval": True,
            "reason": f"Confidence-gated escalation: '{verdict}' verdict signals genuine "
                      f"ambiguity -- exactly where grounding context helps most.",
            "cost_tier": "FULL RETRIEVAL + GENERATION (escalated)",
            "verdict": verdict,
        }


print("=" * 70)
print("MODULE 2: LAYERED TRIGGERING DECISIONS")
print("=" * 70)

trigger_count = 0
skip_count = 0

for label, email in TEST_EMAILS:
    decision = should_trigger_retrieval(email)
    if decision["trigger_retrieval"]:
        trigger_count += 1
    else:
        skip_count += 1
    print(f"\n[{label}]")
    print(f"  Verdict: {decision['verdict']}")
    print(f"  Trigger retrieval? {decision['trigger_retrieval']}")
    print(f"  Reason: {decision['reason']}")
    print(f"  Cost tier: {decision['cost_tier']}")

print(f"\n{trigger_count} of {len(TEST_EMAILS)} queries triggered full retrieval.")
print(f"{skip_count} of {len(TEST_EMAILS)} queries skipped retrieval entirely (FREE).")
print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: LAYERED TRIGGERING DECISIONS

[Clear FD question]
  Verdict: FD
  Trigger retrieval? True
  Reason: Rule-based gate: clear FD verdict, retrieval directly serves this case.
  Cost tier: FULL RETRIEVAL + GENERATION

[Clear Non-FD complaint]
  Verdict: Non-FD
  Trigger retrieval? False
  Reason: Rule-based gate: classified Non-FD, no FD content to look up.
  Cost tier: FREE (rule-based only, no model call)

[Ambiguous mixed case]
  Verdict: Multiple Category
  Trigger retrieval? True
  Reason: Confidence-gated escalation: 'Multiple Category' verdict signals genuine ambiguity -- exactly where grounding context helps most.
  Cost tier: FULL RETRIEVAL + GENERATION (escalated)

[Vague/no signal]
  Verdict: ambiguous
  Trigger retrieval? True
  Reason: Confidence-gated escalation: 'ambiguous' verdict signals genuine ambiguity -- exactly where grounding context helps most.
  Cost tier: FULL RETRIEVAL + GENERATION (escalated)

[Clear FD, senior citizen rate]
  Verdict: FD
  Trigger ret

### Module 3: Measuring the Real Cost Impact of Triggering

Quantify the theory's central cost claim: what fraction of total pipeline cost is actually saved by correct triggering, at realistic production volume.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Quantifying the cost impact of triggering at scale
# ------------------------------------------------------------------

# illustrative per-request cost figures (relative units, not real currency)
COST_RULE_BASED_ONLY = 0.0001    # essentially free -- pure string matching
COST_FULL_RETRIEVAL_AND_GENERATION = 1.0  # full pipeline: retrieval + rerank + generation + verification

# realistic daily volume and verdict distribution (illustrative, based on
# this project's own measured corpus composition: roughly 40% FD,
# 30% Non-FD, 30% Multiple Category/ambiguous)
DAILY_VOLUME = 10_000
VERDICT_DISTRIBUTION = {
    "FD": 0.40,
    "Non-FD": 0.30,
    "Multiple Category": 0.20,
    "ambiguous": 0.10,
}

print("=" * 70)
print("COST IMPACT OF RETRIEVAL TRIGGERING AT PRODUCTION VOLUME")
print("=" * 70)
print(f"Daily volume: {DAILY_VOLUME:,} emails\n")

# Scenario A: NO triggering logic -- retrieval runs on EVERY query
cost_no_triggering = DAILY_VOLUME * COST_FULL_RETRIEVAL_AND_GENERATION

# Scenario B: WITH layered triggering -- Non-FD skips retrieval entirely
non_fd_count = int(DAILY_VOLUME * VERDICT_DISTRIBUTION["Non-FD"])
retrieval_triggered_count = DAILY_VOLUME - non_fd_count

cost_with_triggering = (
    non_fd_count * COST_RULE_BASED_ONLY
    + retrieval_triggered_count * COST_FULL_RETRIEVAL_AND_GENERATION
)

savings = cost_no_triggering - cost_with_triggering
savings_percent = (savings / cost_no_triggering) * 100

print(f"Scenario A -- NO triggering (retrieval on every query):")
print(f"  Total daily cost (relative units): {cost_no_triggering:,.2f}")

print(f"\nScenario B -- WITH layered triggering:")
print(f"  Non-FD queries skipping retrieval: {non_fd_count:,} ({VERDICT_DISTRIBUTION['Non-FD']*100:.0f}% of volume)")
print(f"  Queries still triggering full retrieval: {retrieval_triggered_count:,}")
print(f"  Total daily cost (relative units): {cost_with_triggering:,.2f}")

print(f"\nDaily savings from correct triggering: {savings:,.2f} relative units")
print(f"Percentage cost reduction: {savings_percent:.1f}%")

print("\nThis concretely demonstrates the theory's central cost claim:")
print("simply skipping retrieval for the ~30% of traffic that a FREE")
print("rule-based classifier already confidently identifies as Non-FD")
print("produces a substantial, direct cost reduction -- without touching")
print("anything inside the retrieval pipeline itself.")

print("\nModule 3 complete. All Chapter 9 Topic 1 modules done.")
print()
print("=" * 70)
print("CHAPTER 9 TOPIC 1 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  Retrieval is a DECISION, not a mandatory first step -- something
  upstream must decide whether a query benefits from retrieval before
  the expensive machinery runs.

  Layer the decision: cheap rule-based gate first (skip clearly
  decidable cases for free), confidence-gated escalation second
  (target ambiguous cases specifically), model-decided triggering
  reserved only for what survives the cheaper filters.

  Getting this routing decision right has a LARGER cost impact at
  scale than almost any optimization inside the retrieval pipeline.

  A wrong triggering decision silently determines whether citation,
  faithfulness, and hallucination checks EVER get a chance to run.
""")


COST IMPACT OF RETRIEVAL TRIGGERING AT PRODUCTION VOLUME
Daily volume: 10,000 emails

Scenario A -- NO triggering (retrieval on every query):
  Total daily cost (relative units): 10,000.00

Scenario B -- WITH layered triggering:
  Non-FD queries skipping retrieval: 3,000 (30% of volume)
  Queries still triggering full retrieval: 7,000
  Total daily cost (relative units): 7,000.30

Daily savings from correct triggering: 2,999.70 relative units
Percentage cost reduction: 30.0%

This concretely demonstrates the theory's central cost claim:
simply skipping retrieval for the ~30% of traffic that a FREE
rule-based classifier already confidently identifies as Non-FD
produces a substantial, direct cost reduction -- without touching
anything inside the retrieval pipeline itself.

Module 3 complete. All Chapter 9 Topic 1 modules done.

CHAPTER 9 TOPIC 1 -- KEY POINTS TO REMEMBER

  Retrieval is a DECISION, not a mandatory first step -- something
  upstream must decide whether a query benefits fr